# Custom training loop with `Keras`

In this notebook, we implement custom training loop in `keras`.

We will use the default `model.fit()` method to check our result.  To test our custom training loop, we will use the Fashion MNIST dataset.

In [ ]:
import keras
import tensorflow as tf
import numpy as np

## Default `model.fit()` method

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

We build a simple sequential model and fit it with the default `model.fit()` method.

In [ ]:
neural_net = keras.models.Sequential([
    keras.layers.Flatten(input_shape=(28,28)),
    keras.layers.Dense(100, name="dense", activation="relu"),
    keras.layers.Dense(10, name="output", activation="softmax")
])

In [ ]:
neural_net.summary()

In [ ]:
w, b = neural_net.get_layer('dense').get_weights()

In [ ]:
neural_net.compile(optimizer="sgd",
           loss="sparse_categorical_crossentropy",
           metrics=["accuracy"])

In [ ]:
history = neural_net.fit(X_train, y_train, epochs=5, validation_split=0.2)

Our accuracy isn't great, but that's OK since our focus isn't on predictive performance right now.  Instead, we will build a custom training loop and see if we can reproduce these results.

## Custom training loop

In [ ]:
def random_batch(X, y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]

In [ ]:
# def print_status_bar(iteration, total, loss, metrics=None):
#     message = f"iteration: {iteration} of {total} "
#     end = "" if iteration < total else "\n"
#     if metrics:
#         message += f"\r loss : {loss} - metrics : {metrics}"
#     else:
#         message += f"\r loss : {loss}"
#     print(message, end=end)

# def print_status_bar(iteration, total, loss, metrics=None):
#     metrics = " - ".join(["{}: {:.4f}".format(m.name, m.result())
#                     for m in [loss] + (metrics or [])])
#     end = "" if iteration < total else "\n"
#     print("\r{}/{} - ".format(iteration, total) + metrics, end=end)

In [42]:
# hyperparameters
import math

n_epochs = 5
batch_size = 32
n_steps = math.ceil(len(X_train) / batch_size)
optimizer = keras.optimizers.SGD()
loss_fn = keras.losses.SparseCategoricalCrossentropy()


In [44]:
# training loop
for epoch in range(n_epochs):
    print(f"Epoch {epoch + 1} of {n_epochs}")
    for step in range(n_steps):
        # create mini batches
        start = step * batch_size
        end = min(start + batch_size, len(X_train))
        X_batch = X_train[start:end]
        y_batch = y_train[start:end]
        
        with tf.GradientTape() as tape:
            y_pred = neural_net(X_batch, training=True)
            loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss_val = loss.numpy()
        gradients = tape.gradient(loss, neural_net.trainable_variables)
        optimizer.apply_gradients(zip(gradients, neural_net.trainable_variables))
        # print_status_bar(step*batch_size, len(y_train), loss_val)
    

Epoch 1 of 5


KeyboardInterrupt: 

In [ ]:
y_training_prob = neural_net(X_train, training=False)
y_training_pred = np.argmax(y_training_prob, axis=1)

In [ ]:
acc = keras.metrics.Accuracy()
acc(y_train, y_training_pred)